# Colombia (COP) — Fixed Income Derivatives

IBR swaps, TES fixed-rate bonds, TES UVR (inflation-linked).

**Key features:**
- IBR: Indicador Bancario de Referencia (BanRep overnight rate)
- TES: Títulos de Tesorería (annual coupon, ACT/365)
- UVR: Unidad de Valor Real — daily inflation unit

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.colombian import (
    IBRSwap, TESBond, TESUVRBond,
    build_ibr_curve, synthetic_ibr_strip,
)
from pricebook.fixed_income.inflation_unit import InflationUnitBond, dual_curve_breakeven
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"IBR policy rate: 9.75% (BanRep)")
print(f"UVR value: ~350 COP")

## 1. IBR Curve Construction

In [ ]:
strip = synthetic_ibr_strip(REF, ibr=0.0975)
ibr_curve = build_ibr_curve(REF, strip)

print(f"IBR Swap Strip ({len(strip)} points):")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in strip:
    print(f"{c['months']:>6}M  {c['rate']*100:>7.2f}%  {ibr_curve.df(c['maturity']):>10.6f}")

with apply_theme():
    fig, (ax1, ax2) = create_figure(2)
    tenors = [c["years"] for c in strip]
    ax1.plot(tenors, [c["rate"]*100 for c in strip], 'o-', linewidth=2)
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("IBR Rate (%)")
    ax1.set_title("IBR Swap Curve — Nov 2024")
    ax1.axhline(9.75, color='gray', linestyle='--', alpha=0.5, label='Policy 9.75%')
    ax1.legend()

    ax2.plot(tenors, [ibr_curve.df(c["maturity"]) for c in strip], 's-', linewidth=2, color='#059669')
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Discount Factor")
    ax2.set_title("IBR Discount Factors")
    fig.tight_layout()

## 2. IBR Swap Pricing

In [ ]:
print(f"IBR Swap Pricing (COP 100bn notional):")
print(f"{'Tenor':>6}  {'Par Rate':>10}  {'DV01 (COP)':>14}")
print(f"{'─'*6}  {'─'*10}  {'─'*14}")
for t in [1, 2, 3, 5, 7, 10]:
    swap = IBRSwap(REF, REF + relativedelta(years=t), 0.0975, notional=100e9)
    r = swap.price(ibr_curve)
    print(f"{t:>4}Y  {r.par_rate*100:>9.2f}%  {r.dv01:>14,.0f}")

## 3. TES Fixed-Rate Bond

In [ ]:
print(f"TES Bond Pricing (annual coupon, ACT/365):")
print(f"{'Coupon':>8}  {'Tenor':>6}  {'Dirty Price':>12}")
print(f"{'─'*8}  {'─'*6}  {'─'*12}")
for cpn in [0.07, 0.09, 0.11]:
    for t in [3, 5, 10, 20]:
        tes = TESBond(REF, REF + relativedelta(years=t), cpn)
        px = tes.dirty_price(ibr_curve)
        print(f"{cpn*100:>7.1f}%  {t:>4}Y  {px:>11.4f}")
    print()

## 4. TES UVR — Inflation-Linked Bond

In [ ]:
# UVR real curve at ~3.5%
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod

uvr_dates = [REF + relativedelta(years=y) for y in [1, 2, 3, 5, 10, 20]]
uvr_dfs = [math.exp(-0.035 * y) for y in [1, 2, 3, 5, 10, 20]]
uvr_curve = DiscountCurve(REF, uvr_dates, uvr_dfs, interpolation=InterpolationMethod.LOG_LINEAR)

current_uvr = 352.47

print(f"TES UVR Pricing (UVR = {current_uvr:.2f} COP):")
print(f"{'Tenor':>6}  {'Real Cpn':>10}  {'Real PV':>10}  {'Nominal PV':>12}  {'Real Yield':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*10}")
for t, cpn in [(3, 0.03), (5, 0.035), (10, 0.035), (20, 0.04)]:
    uvr_bond = TESUVRBond(REF, REF + relativedelta(years=t), cpn)
    r = uvr_bond.price(REF, uvr_curve, current_uvr)
    print(f"{t:>4}Y  {cpn*100:>9.2f}%  {r.real_price:>10.4f}  {r.nominal_price:>12,.0f}  {r.real_yield*100:>9.2f}%")

# BEI
bei = dual_curve_breakeven(ibr_curve, uvr_curve, [1, 2, 3, 5, 10, 20])
print(f"\nBreakeven Inflation:")
for row in bei:
    print(f"  {row['years']:>2}Y: {row['bei']*100:.2f}%")

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| IBR Swap | ACT/360, overnight | BanRep reference rate |
| TES | Annual, ACT/365 | Fixed-rate sovereign |
| TES UVR | Annual, UVR-linked | Daily inflation unit |
| BEI | IBR - UVR | ~6% implied inflation |